In [1]:
import pickle
import pandas as pd
import gzip
import re
import random
import numpy as np
from tqdm import tqdm

# 设置随机种子保证可重现性
random.seed(42)
np.random.seed(42)

In [ ]:
# load datasets
rec_train = pickle.load(open('./qilin_rerank/ori_data/rec_train.pkl', 'rb'))
src_train = pickle.load(open('./qilin_rerank/ori_data/src_train.pkl', 'rb'))

src_train.head()

,user_idx,gt_note_idx,neg_note_idx,timestamp,history,task,query
0,19,"[978956, 979823]","[681566, 872489]",1.732667e+09,"[1584325, 965158, 1702543, 967780, 1720648, 13...",search,表白话术对男生
1,19,[],[1272140],1.732667e+09,"[1162312, 1221274, 979823, 978956, 1584325, 96...",search,网恋表白开口
2,19,[979823],"[1272140, 681566]",1.732667e+09,"[1162312, 1221274, 979823, 978956, 1584325, 96...",search,网恋表白话术
3,19,"[979823, 1531849]",[892863],1.732668e+09,"[979823, 1162312, 1221274, 979823, 978956, 158...",search,表白话术对男生
4,19,[825317],[1217615],1.732668e+09,[979823],search,表白男生真诚


In [3]:
# 首先收集所有的item来构建item corpus
def collect_all_items(train_data_list):
    """收集所有数据集中的items来构建item corpus"""
    item_set = set()
    
    for data in train_data_list:
        for _, row in data.iterrows():
            # 添加正样本
            if isinstance(row['gt_note_idx'], list):
                for item in row['gt_note_idx']:
                    item_set.add(item)
            else:
                item_set.add(row['gt_note_idx'])
            
            # 添加历史交互物品
            if isinstance(row['history'], list):
                for item in row['history']:
                    item_set.add(item)
            
            # 添加原有的负样本（保证覆盖）
            if isinstance(row['neg_note_idx'], list):
                for item in row['neg_note_idx']:
                    item_set.add(item)
    
    return list(item_set)

def sample_negative_items(user_history, target_item, item_corpus, num_neg=99):
    """为单个用户采样负样本
    
    Args:
        user_history: 用户历史交互物品列表
        target_item: 当前正样本物品
        item_corpus: 所有物品的列表
        num_neg: 负样本数量
    
    Returns:
        负样本物品ID列表
    """
    # 创建用户已交互物品集合（包括历史和当前正样本）
    user_items = set(user_history + [target_item])
    
    # 从item corpus中排除用户已交互的物品
    candidate_items = [item for item in item_corpus if item not in user_items]
    
    # 随机采样负样本
    if len(candidate_items) < num_neg:
        print(f"Warning: 候选负样本数量({len(candidate_items)})少于需求数量({num_neg})")
        return candidate_items
    
    negative_items = random.sample(candidate_items, num_neg)
    return negative_items

# 收集所有items构建item corpus
print("正在构建item corpus...")
all_datasets = [rec_train, src_train]
item_corpus = collect_all_items(all_datasets)
print(f"Item corpus大小: {len(item_corpus)}")


正在构建item corpus...
Item corpus大小: 266988


In [4]:
# Split gt_note_idx lists into individual rows to expand the dataset
print("正在处理src_train数据并重新采样负样本...")
expanded_data = []
for _, row in tqdm(src_train.iterrows(), total=len(src_train), desc="处理src_train"):
    for gt_note in row['gt_note_idx']:
        new_row = row.copy()
        new_row['gt_note_idx'] = gt_note
        
        # 重新采样负样本
        user_history = row['history'] if isinstance(row['history'], list) else []
        new_neg_samples = sample_negative_items(user_history, gt_note, item_corpus, num_neg=99)
        new_row['neg_note_idx'] = new_neg_samples
        
        expanded_data.append(new_row)

src_train = pd.DataFrame(expanded_data)
# change name of gt_note_idx to target_item
src_train.rename(columns={'gt_note_idx': 'target_item'}, inplace=True)
# change history to item_list
src_train.rename(columns={'history': 'item_list'}, inplace=True)
# change user_idx to user_id
src_train.rename(columns={'user_idx': 'user_id'}, inplace=True)
# reset index
src_train.reset_index(drop=True, inplace=True)
print(f"src_train处理完成，共{len(src_train)}条数据")
src_train.head()


正在处理src_train数据并重新采样负样本...


处理src_train: 100%|██████████| 13185/13185 [05:21<00:00, 41.05it/s]


src_train处理完成，共12566条数据


,user_id,target_item,neg_note_idx,timestamp,item_list,task,query
0,19,978956,"[867521, 1457838, 1337176, 1361413, 1971335, 8...",1.732667e+09,"[1584325, 965158, 1702543, 967780, 1720648, 13...",search,表白话术对男生
1,19,979823,"[1114449, 397348, 1250038, 1270597, 788740, 14...",1.732667e+09,"[1584325, 965158, 1702543, 967780, 1720648, 13...",search,表白话术对男生
2,19,979823,"[1459888, 1159071, 1311323, 602565, 981019, 16...",1.732667e+09,"[1162312, 1221274, 979823, 978956, 1584325, 96...",search,网恋表白话术
3,19,979823,"[1939412, 1459266, 1473856, 1003183, 1737517, ...",1.732668e+09,"[979823, 1162312, 1221274, 979823, 978956, 158...",search,表白话术对男生
4,19,1531849,"[959587, 1467106, 1514738, 1349120, 1081423, 1...",1.732668e+09,"[979823, 1162312, 1221274, 979823, 978956, 158...",search,表白话术对男生


In [5]:
# do the same thing for rec_train
print("正在处理rec_train数据并重新采样负样本...")
expanded_data = []
for _, row in tqdm(rec_train.iterrows(), total=len(rec_train), desc="处理rec_train"):
    for gt_note in row['gt_note_idx']:
        new_row = row.copy()
        new_row['gt_note_idx'] = gt_note
        
        # 重新采样负样本
        user_history = row['history'] if isinstance(row['history'], list) else []
        new_neg_samples = sample_negative_items(user_history, gt_note, item_corpus, num_neg=99)
        new_row['neg_note_idx'] = new_neg_samples
        
        expanded_data.append(new_row)

rec_train = pd.DataFrame(expanded_data)
# change name of gt_note_idx to target_item
rec_train.rename(columns={'gt_note_idx': 'target_item'}, inplace=True)
# change history to item_list
rec_train.rename(columns={'history': 'item_list'}, inplace=True)
# change user_idx to user_id
rec_train.rename(columns={'user_idx': 'user_id'}, inplace=True)
# reset index
rec_train.reset_index(drop=True, inplace=True)
print(f"rec_train处理完成，共{len(rec_train)}条数据")
rec_train.head()


正在处理rec_train数据并重新采样负样本...


处理rec_train: 100%|██████████| 46870/46870 [37:14<00:00, 20.98it/s]  


rec_train处理完成，共88887条数据


,user_id,target_item,neg_note_idx,timestamp,item_list,task,query
0,7,1394473,"[1629422, 1103668, 953532, 998735, 1120929, 10...",1.732665e+09,"[886649, 830604, 978573, 1635358, 1088587, 115...",rec,NaN
1,7,1403838,"[1939435, 919495, 823187, 1409559, 999416, 141...",1.732665e+09,"[886649, 830604, 978573, 1635358, 1088587, 115...",rec,NaN
2,7,785527,"[836330, 899138, 1113520, 1261126, 1082154, 85...",1.732680e+09,"[1403838, 1394473, 886649, 830604, 978573, 163...",rec,NaN
3,7,981107,"[1580292, 1478219, 1979139, 550344, 1430222, 9...",1.732680e+09,"[1403838, 1394473, 886649, 830604, 978573, 163...",rec,NaN
4,7,1019287,"[1242279, 1489957, 903150, 831825, 1702490, 11...",1.732681e+09,"[1671203, 815425, 981107, 785527, 1403838, 139...",rec,NaN


In [ ]:
train = pd.concat([rec_train, src_train], ignore_index=True)
train.sort_values(by=['user_id', 'timestamp'], inplace=True)
train.reset_index(drop=True, inplace=True)

train_dict = train.to_dict(orient='index')
pickle.dump(train_dict, open('./qilin_rerank/seq_data/train.pkl', 'wb'))

rec_train_dict = rec_train.to_dict(orient='index')
pickle.dump(rec_train_dict, open('./qilin_rerank/seq_data/train_rec.pkl', 'wb'))

# transform to dict and save
src_train_dict = src_train.to_dict(orient='index')
pickle.dump(src_train_dict, open('./qilin_rerank/seq_data/train_src.pkl', 'wb'))

In [ ]:
# do the same thing for valid
# load datasets
rec_valid = pickle.load(open('./qilin_rerank/ori_data/rec_valid.pkl', 'rb'))
src_valid = pickle.load(open('./qilin_rerank/ori_data/src_valid.pkl', 'rb'))

# 更新item corpus包含验证集的items
print("更新item corpus包含验证集数据...")
valid_datasets = [rec_valid, src_valid]
valid_items = collect_all_items(valid_datasets)
item_corpus.extend(valid_items)
item_corpus = list(set(item_corpus))  # 去重
print(f"更新后的Item corpus大小: {len(item_corpus)}")

print("正在处理rec_valid数据并重新采样负样本...")
expanded_data = []
for _, row in tqdm(rec_valid.iterrows(), total=len(rec_valid), desc="处理rec_valid"):
    for gt_note in row['gt_note_idx']:
        new_row = row.copy()
        new_row['gt_note_idx'] = gt_note
        
        # 重新采样负样本
        user_history = row['history'] if isinstance(row['history'], list) else []
        new_neg_samples = sample_negative_items(user_history, gt_note, item_corpus, num_neg=99)
        new_row['neg_note_idx'] = new_neg_samples
        
        expanded_data.append(new_row)

rec_valid = pd.DataFrame(expanded_data)
# change name of gt_note_idx to target_item
rec_valid.rename(columns={'gt_note_idx': 'target_item'}, inplace=True)
# change history to item_list
rec_valid.rename(columns={'history': 'item_list'}, inplace=True)
# change user_idx to user_id
rec_valid.rename(columns={'user_idx': 'user_id'}, inplace=True)
# reset index
rec_valid.reset_index(drop=True, inplace=True)
print(f"rec_valid处理完成，共{len(rec_valid)}条数据")

# do the same thing for src_valid
print("正在处理src_valid数据并重新采样负样本...")
expanded_data = []
for _, row in tqdm(src_valid.iterrows(), total=len(src_valid), desc="处理src_valid"):
    for gt_note in row['gt_note_idx']:
        new_row = row.copy()
        new_row['gt_note_idx'] = gt_note
        
        # 重新采样负样本
        user_history = row['history'] if isinstance(row['history'], list) else []
        new_neg_samples = sample_negative_items(user_history, gt_note, item_corpus, num_neg=99)
        new_row['neg_note_idx'] = new_neg_samples
        
        expanded_data.append(new_row)

src_valid = pd.DataFrame(expanded_data)
# change name of gt_note_idx to target_item
src_valid.rename(columns={'gt_note_idx': 'target_item'}, inplace=True)
# change history to item_list
src_valid.rename(columns={'history': 'item_list'}, inplace=True)
# change user_idx to user_id
src_valid.rename(columns={'user_idx': 'user_id'}, inplace=True)
# reset index
src_valid.reset_index(drop=True, inplace=True)
print(f"src_valid处理完成，共{len(src_valid)}条数据")

valid = pd.concat([rec_valid, src_valid], ignore_index=True)
valid.sort_values(by=['user_id', 'timestamp'], inplace=True)
valid.reset_index(drop=True, inplace=True)

valid_dict = valid.to_dict(orient='index')
pickle.dump(valid_dict, open('./qilin_rerank/seq_data/valid.pkl', 'wb'))

rec_valid_dict = rec_valid.to_dict(orient='index')
pickle.dump(rec_valid_dict, open('./qilin_rerank/seq_data/valid_rec.pkl', 'wb'))

src_valid_dict = src_valid.to_dict(orient='index')
pickle.dump(src_valid_dict, open('./qilin_rerank/seq_data/valid_src.pkl', 'wb'))



更新item corpus包含验证集数据...
更新后的Item corpus大小: 274734
正在处理rec_valid数据并重新采样负样本...


处理rec_valid: 100%|██████████| 3790/3790 [03:38<00:00, 17.31it/s]


rec_valid处理完成，共8408条数据
正在处理src_valid数据并重新采样负样本...


处理src_valid: 100%|██████████| 2409/2409 [00:33<00:00, 70.95it/s] 


src_valid处理完成，共1651条数据


In [ ]:
# do the same thing for test
rec_test = pickle.load(open('./qilin_rerank/ori_data/rec_test.pkl', 'rb'))
src_test = pickle.load(open('./qilin_rerank/ori_data/src_test.pkl', 'rb'))

# 更新item corpus包含测试集的items
print("更新item corpus包含测试集数据...")
test_datasets = [rec_test, src_test]
test_items = collect_all_items(test_datasets)
item_corpus.extend(test_items)
item_corpus = list(set(item_corpus))  # 去重
print(f"最终Item corpus大小: {len(item_corpus)}")

# 为测试集重新采样负样本
print("正在为rec_test重新采样负样本...")
for idx, row in tqdm(rec_test.iterrows(), total=len(rec_test), desc="处理rec_test"):
    user_history = row['history'] if isinstance(row['history'], list) else []
    # 测试集的target_item是列表格式，需要排除所有正样本
    target_items = row['gt_note_idx'] if isinstance(row['gt_note_idx'], list) else [row['gt_note_idx']]
    
    # 排除用户历史和所有正样本
    all_user_items = set(user_history + target_items)
    candidate_items = [item for item in item_corpus if item not in all_user_items]
    
    # 采样负样本
    if len(candidate_items) >= 99:
        new_neg_samples = random.sample(candidate_items, 99)
    else:
        new_neg_samples = candidate_items
        print(f"Warning: rec_test第{idx}行候选负样本数量({len(candidate_items)})少于99")
    
    rec_test.at[idx, 'neg_note_idx'] = new_neg_samples

print("正在为src_test重新采样负样本...")
for idx, row in tqdm(src_test.iterrows(), total=len(src_test), desc="处理src_test"):
    user_history = row['history'] if isinstance(row['history'], list) else []
    # 测试集的target_item是列表格式，需要排除所有正样本
    target_items = row['gt_note_idx'] if isinstance(row['gt_note_idx'], list) else [row['gt_note_idx']]
    
    # 排除用户历史和所有正样本
    all_user_items = set(user_history + target_items)
    candidate_items = [item for item in item_corpus if item not in all_user_items]
    
    # 采样负样本
    if len(candidate_items) >= 99:
        new_neg_samples = random.sample(candidate_items, 99)
    else:
        new_neg_samples = candidate_items
        print(f"Warning: src_test第{idx}行候选负样本数量({len(candidate_items)})少于99")
    
    src_test.at[idx, 'neg_note_idx'] = new_neg_samples

# change name of gt_note_idx to target_item
rec_test.rename(columns={'gt_note_idx': 'target_item'}, inplace=True)
# change history to item_list
rec_test.rename(columns={'history': 'item_list'}, inplace=True)
# change user_idx to user_id
rec_test.rename(columns={'user_idx': 'user_id'}, inplace=True)

# do the same thing for src_test
src_test.rename(columns={'gt_note_idx': 'target_item'}, inplace=True)
# change history to item_list
src_test.rename(columns={'history': 'item_list'}, inplace=True)
# change user_idx to user_id
src_test.rename(columns={'user_idx': 'user_id'}, inplace=True)

# reset index
rec_test.reset_index(drop=True, inplace=True)
src_test.reset_index(drop=True, inplace=True)
print(f"rec_test处理完成，共{len(rec_test)}条数据")
print(f"src_test处理完成，共{len(src_test)}条数据")

test = pd.concat([rec_test, src_test], ignore_index=True)
test.sort_values(by=['user_id', 'timestamp'], inplace=True)
test.reset_index(drop=True, inplace=True)

test_dict = test.to_dict(orient='index')
pickle.dump(test_dict, open('./qilin_rerank/seq_data/test.pkl', 'wb'))

rec_test_dict = rec_test.to_dict(orient='index')
pickle.dump(rec_test_dict, open('./qilin_rerank/seq_data/test_rec.pkl', 'wb'))

src_test_dict = src_test.to_dict(orient='index')
pickle.dump(src_test_dict, open('./qilin_rerank/seq_data/test_src.pkl', 'wb'))


更新item corpus包含测试集数据...
最终Item corpus大小: 275526
正在为rec_test重新采样负样本...


处理rec_test: 100%|██████████| 3790/3790 [01:32<00:00, 40.92it/s]


正在为src_test重新采样负样本...


处理src_test: 100%|██████████| 2409/2409 [01:15<00:00, 31.89it/s]


rec_test处理完成，共3790条数据
src_test处理完成，共2409条数据


In [ ]:
# get item number and user number
# open train_src.pkl
import pickle
item_set = set()
user_set = set()

with open('./qilin_rerank/seq_data/train.pkl', 'rb') as f:
    train_src = pickle.load(f)
# iterate the dict and get the data
for k, v in train_src.items():
    for item in v['item_list']:
        item_set.add(item)
    item_set.add(v['target_item'])
    user_set.add(v['user_id'])
    for item in v['neg_note_idx']:
        item_set.add(item)

# do the same for test
with open('./qilin_rerank/seq_data/test.pkl', 'rb') as f:
    test_src = pickle.load(f)
for k, v in test_src.items():
    for item in v['item_list']:
        item_set.add(item)
    for item in v['neg_note_idx']:
        item_set.add(item)
    for item in v['target_item']:
        item_set.add(item)
    user_set.add(v['user_id'])
    

# do the same for valid
with open('./qilin_rerank/seq_data/valid.pkl', 'rb') as f:
    valid_src = pickle.load(f)
for k, v in valid_src.items():
    for item in v['item_list']:
        item_set.add(item)
    item_set.add(v['target_item'])
    user_set.add(v['user_id'])
    for item in v['neg_note_idx']:
        item_set.add(item)

print('item_number')
print(len(item_set))
print('user_number')
print(len(user_set))

# get the data number for each set
print('train search interaction number')
print(len(train_src))
print('test search interaction number')
print(len(test_src))
print('valid search interaction number')
print(len(valid_src))


item_number
275515
user_number
3816
train search interaction number
101453
test search interaction number
6199
valid search interaction number
10059


In [ ]:
import csv

with open('./qilin_rerank/raw_data/notes.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    # convert to pandas dataframe (first line is header)
    item_meta = pd.DataFrame(list(reader))
    item_meta.columns = item_meta.iloc[0]
    item_meta = item_meta.iloc[1:]
item_meta.head()



,note_title,note_content,note_type,video_duration,video_height,video_width,image_num,content_length,commercial_flag,taxonomy1_id,...,accum_like_num,accum_collect_num,accum_comment_num,view_time,rec_view_time,search_view_time,valid_view_times,full_view_times,note_idx,image_path
1,这题我会！每年都要灌几十斤的肉！,每年只要到灌香肠的环节，就说明快要过年了🧨[偷笑R]有一种让人幸福的仪式感！家里只有我会灌，...,2.0,227.0,1920.0,1080.0,1.0,89.0,0.0,5ab094be481d26b8ef9045f1,...,13022.0,7829.0,414.0,253719.0,0.0,246595.0,3769.0,347.0,0,['image/part_114/1147826/5133571.jpg' 'image/p...
2,为什么有的孩子补课越补越差，真相很扎心,\n补课真的有用吗？为什么有的孩子越补越好，而有的孩子怎么补也不行呢？一个扎心的事实是补课只...,1.0,0.0,0.0,0.0,1.0,1006.0,0.0,5ab094be481d26b8ef904608,...,22.0,9.0,7.0,0.0,0.0,0.0,0.0,0.0,1,['image/part_115/1151325/5149856.jpg']
3,🇲🇾KL的小姐姐，这样的饺子你吃过吗？,一：煮饺子（只需10分钟）\n1.无需解冻，水开后直接下锅，\n2.等待水将饺子煮开。\n3...,1.0,0.0,0.0,0.0,3.0,489.0,0.0,5ab094be481d26b8ef9045f1,...,2687.0,1114.0,17.0,0.0,0.0,0.0,0.0,0.0,2,['image/part_114/1140491/5100533.jpg' 'image/p...
4,新加坡女佣攻略手把手！省几千中介费不是梦,趁自己还记得赶紧写下来给姐妹们参考。\n这篇主要写一下转女佣(字数刚好一千，言简意赅)的教程...,1.0,0.0,0.0,0.0,12.0,920.0,0.0,c00000000000000000000301,...,126.0,216.0,55.0,0.0,0.0,0.0,0.0,0.0,3,['image/part_116/1165079/5214270.jpg' 'image/p...
5,不敢想象生日收到这个我能有多开朗🥹,没有人比这个哥更会送礼了\n不敢想象这个女孩子收到这个礼物该有多快乐\n\t\n#生日礼物开...,1.0,0.0,0.0,0.0,7.0,122.0,0.0,c00000000000000000000558,...,3792.0,983.0,538.0,0.0,0.0,0.0,0.0,0.0,4,['image/part_117/1176128/5265862.jpg' 'image/p...


In [11]:
# filter the item_meta df according to item_set, and save new one
# 将item_meta转换为字典格式,key为note_idx,value为该note的所有信息
item_meta_dict = {}
for idx, row in item_meta.iterrows():
    # if is int
    try:
        item_meta_dict[int(row['note_idx'])] = row.to_dict()
    except:
        continue

# 根据item_set过滤item_meta_dict,只保留在item_set中的item
new_item_meta = {}
for item_id in item_set:
    if item_id in item_meta_dict:
        new_item_meta[item_id] = item_meta_dict[item_id]


In [ ]:
article_text = []
item_id = []
cnt = 0
for key, value in new_item_meta.items():
    text = ''
    if isinstance(value.get('note_title'), str) and not pd.isna(value['note_title']):
        text += 'title: ' + value['note_title'] + ' '
    if isinstance(value.get('note_content'), str) and not pd.isna(value['note_content']):
        text += 'content: ' + value['note_content'] + ' '
    text = text.strip()
    text = re.sub(r'\n', '', text)
    article_text.append(text)
    item_id.append(key)
    cnt += 1

# cehck whether there is same article 
uniq_article = set(article_text)
print(f'Number of unique articles: {len(uniq_article)}')
print(f'Number of All articles: {len(article_text)}')

with open('./qilin_rerank/seq_data/item_plain_text.txt', 'w') as f:
    for item, text in zip(item_id, article_text):
        output = str(item) + ' ' + text + '\n'
        f.write(output)

Number of unique articles: 76955
Number of All articles: 77135


In [13]:
# 验证新的负样本采样效果
print("="*50)
print("负样本采样验证")
print("="*50)

# 检查训练集
print("\n检查训练集负样本...")
with open('train.pkl', 'rb') as f:
    train_data = pickle.load(f)

sample_idx = 0
sample_data = train_data[sample_idx]
print(f"样本 {sample_idx}:")
print(f"用户ID: {sample_data['user_id']}")
print(f"目标物品: {sample_data['target_item']}")
print(f"历史物品数量: {len(sample_data['item_list'])}")
print(f"负样本数量: {len(sample_data['neg_note_idx'])}")

# 检查负样本是否与用户历史重叠
user_items = set(sample_data['item_list'] + [sample_data['target_item']])
neg_items = set(sample_data['neg_note_idx'])
overlap = user_items.intersection(neg_items)
print(f"负样本与用户交互物品重叠数量: {len(overlap)}")
if len(overlap) > 0:
    print(f"重叠物品: {overlap}")
else:
    print("✓ 负样本与用户历史无重叠")

print(f"\n前10个负样本: {sample_data['neg_note_idx'][:10]}")

print("\n" + "="*50)
print("负样本采样修改完成！")
print("新的采样逻辑:")
print("1. 从完整的item corpus中采样99个负样本")
print("2. 排除用户历史交互的所有物品")
print("3. 排除当前的正样本物品")
print("4. 随机采样保证负样本的多样性")
print("="*50)


负样本采样验证

检查训练集负样本...
样本 0:
用户ID: 7
目标物品: 1394473
历史物品数量: 0
负样本数量: 99
负样本与用户交互物品重叠数量: 0
✓ 负样本与用户历史无重叠

前10个负样本: [768890, 77142, 1815608, 1225088, 815133, 1860663, 1363337, 1318781, 729784, 1895058]

负样本采样修改完成！
新的采样逻辑:
1. 从完整的item corpus中采样99个负样本
2. 排除用户历史交互的所有物品
3. 排除当前的正样本物品
4. 随机采样保证负样本的多样性
